In [1]:
# Run once — comment out after first install
import sys, subprocess
packages = [
    "langchain", "langchain-text-splitters",
    "sentence-transformers", "faiss-cpu",
    "google-generativeai", "google-genai",
    "networkx", "pdfplumber",
]
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
print("✅ All packages installed.")

✅ All packages installed.


In [2]:
import sys, os
from pathlib import Path

# Project root — wherever this notebook lives
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH  = PROJECT_ROOT / "data_legal"
INDEX_PATH = DATA_PATH / "indexes"
INDEX_PATH.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"src found    : {(PROJECT_ROOT / 'src').exists()}")
print(f"data_legal   : {DATA_PATH.exists()}")
print(f"indexes dir  : {INDEX_PATH}")

PROJECT_ROOT : C:\Users\Vohita\RAG
src found    : True
data_legal   : True
indexes dir  : C:\Users\Vohita\RAG\data_legal\indexes


In [3]:
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")
if not GEMINI_API_KEY:
    raise EnvironmentError("GEMINI_API_KEY not set.")
print("✅ API key loaded.")

✅ API key loaded.


In [4]:
from sentence_transformers import SentenceTransformer, CrossEncoder
from src.llm.llm_client import GeminiClient

print("Loading embedding model...")
embed_model  = SentenceTransformer("all-MiniLM-L6-v2")

print("Loading reranker model...")
rerank_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# Single LLM client — shared across ALL modules via injection
llm = GeminiClient()   # reads GEMINI_API_KEY from environment

print("✅ All models ready.")
# # Smoke test
# print(llm.generate("In one sentence, what is a legal contract?"))

Loading embedding model...
Loading reranker model...
✅ All models ready.


In [5]:
from src.agent.scope_guard   import enforce_legal_scope
from src.agent.intent_classifier import classify_intent

# Quick self-test
tests = [
    "What does the non-compete clause say?",
    "Can I advertise the software?",
    "What is Territory defined as?",
    "What must the Licensee do?",
    "Who indemnifies whom?",
    "give gist of the document"
]
for q in tests:
    print(f"{classify_intent(q):25s}  {q}")

general_legal_query        What does the non-compete clause say?
permission_check           Can I advertise the software?
definition_lookup          What is Territory defined as?
obligation_analysis        What must the Licensee do?
general_legal_query        Who indemnifies whom?
summary_request            give gist of the document


In [6]:
from src.ingestion import ingest_and_chunk_document

# ── Change this path to your legal document (.txt or .pdf) ───────────────────
LEGAL_FILE = r"C:\Users\Vohita\Downloads\CUAD_v1-20260117T070211Z-1-001\CUAD_v1\full_contract_txt\GlobalTechnologiesGroupInc_20050928_10KSB_EX-10.9_4148808_EX-10.9_Content License Agreement.txt"

legal_chunks_df = ingest_and_chunk_document(LEGAL_FILE)
legal_chunks_df.head(2)

Ingested : 58 chunks
Avg len  : 742 chars
Min/Max  : 14 / 1022 chars


,chunk_id,chunk_text,domain
0,legal_chunk_0,"EXHIBIT 10.9 GLOBAL MUSIC INTERNATIONAL, I...",legal
1,legal_chunk_1,"Distributor's authorized signature, is REQUIRE...",legal


In [7]:
from src.memory.session_memory import SessionState

# Create session and store models inside it immediately
# so no function ever needs to receive them as separate args
session              = SessionState()
session.embed_model  = embed_model
session.rerank_model = rerank_model

print("Session created. Pipeline ready:", session.is_pipeline_ready())
# (will be False until we load chunks + indexes below)

Session created. Pipeline ready: False


In [8]:
print("Before indexing:", len(legal_chunks_df))

Before indexing: 58


In [9]:
from src.retrieval.indexer import build_legal_search_indexes

bm25_engine, faiss_engine, final_chunks_df = build_legal_search_indexes(
    chunks_df   = legal_chunks_df,
    embed_model = embed_model,
    index_path  = INDEX_PATH,
)

# Store everything in session
session.chunks_df    = final_chunks_df
session.bm25_engine  = bm25_engine
session.faiss_engine = faiss_engine

print(f"Chunks     : {len(final_chunks_df)}")
print(f"BM25 docs  : {bm25_engine.N}")
print(f"FAISS vecs : {faiss_engine.ntotal}")
print(f"Pipeline ready: {session.is_pipeline_ready()}")

Building BM25 index...
Building FAISS index...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Indexed 58 chunks -> C:\Users\Vohita\RAG\data_legal\indexes
Chunks     : 58
BM25 docs  : 58
FAISS vecs : 58
Pipeline ready: True


In [19]:
from src.document_analysis import DocumentAnalyzer

analyzer = DocumentAnalyzer(llm_client=llm, bm25_engine=bm25_engine)
insights = analyzer.analyze_document()

# Populate session with insights — now accessible everywhere
session.populate_insights(insights)

import json
print(json.dumps(insights, indent=2))

{
  "summary": "An agreement for the distribution of music video content via wireless networks in China, detailing operational responsibilities, payment terms, confidentiality, and liabilities between a content provider (IMNTV) and a distributor (MobileVision).",
  "parties": [
    "GLOBAL MUSIC INTERNATIONAL, INC. D/B/A INDEPENDENT MUSIC NETWORK (IMNTV)",
    "MOBILEVISION COMMUNICATIONS LTD. (DISTRIBUTOR)"
  ],
  "agreement_date": "July 13, 2005",
  "agreement_duration": "The initial term begins on the Effective Date (July 13, 2005) and ends twelve (12) months after the 'Launch' (date not specified). The agreement may be extended for additional one-year terms upon mutual agreement and satisfactory performance by both parties. There is also a pilot commercial program for two to three months where content is provided without charge.",
  "termination_clause": "Either party may terminate the Agreement for a material breach by the other party that remains uncured after thirty (30) days wr

In [11]:
from src.kg.build_legal_kg   import build_legal_kg, save_graph
from src.kg.kg_query_engine  import summarize_graph, query_policy

kg = build_legal_kg(final_chunks_df)
save_graph(kg)

# Store in session so executor can query it every turn
session.kg = kg

print(summarize_graph(kg))
print("\nClauses governing 'advertising':", query_policy(kg, "advertising"))
print("Clauses governing 'termination':", query_policy(kg, "termination"))


Knowledge graph saved: 67 nodes, 90 edges -> legal_kg.pkl
{'total_nodes': 67, 'total_edges': 90, 'by_type': {'clause': 58, 'domain': 1, 'obligation': 2, 'policy': 5, 'restriction': 1}}

Clauses governing 'advertising': ['legal_chunk_19', 'legal_chunk_52']
Clauses governing 'termination': ['legal_chunk_26', 'legal_chunk_28', 'legal_chunk_29', 'legal_chunk_30', 'legal_chunk_31', 'legal_chunk_32', 'legal_chunk_33', 'legal_chunk_44']


In [12]:
from src.agent.scope_guard       import enforce_legal_scope
from src.agent.intent_classifier import classify_intent

tests = [
    "What does the non-compete clause say?",
    "Can I advertise the software?",
    "What is Territory?",
    "What must the Licensee do?",
    "Who indemnifies whom?",
    # "Summarize the agreement",
    "is the distributor prohibited from using Programming and IMNTV's Marks to market, advertise, and promote the Programming in the Mobile Software Application(s?"
]
for q in tests:
    print(f"{classify_intent(q):25s}  →  {q}")

general_legal_query        →  What does the non-compete clause say?
permission_check           →  Can I advertise the software?
definition_lookup          →  What is Territory?
obligation_analysis        →  What must the Licensee do?
general_legal_query        →  Who indemnifies whom?
general_legal_query        →  is the distributor prohibited from using Programming and IMNTV's Marks to market, advertise, and promote the Programming in the Mobile Software Application(s?


In [13]:
# from src.agent.executor import run_agent
# from src.agent.scope_guard import enforce_legal_scope

# def ask(question: str) -> str:
#     """
#     Single entry point for any legal query.
#     This exact function will be called from the Streamlit UI.
#     """
#     try:
#         enforce_legal_scope(question)
#     except ValueError as e:
#         return str(e)
    
#     return run_agent(question, session, llm)


# # Test questions
# questions = [
#     "What is the advertising policy in the software agreement?",
#     "Can the agreement be extended after the initial term?",
#     "Is the distributor prohibited from using IMNTV Marks to market the software?",
#     "Who indemnifies whom and what liabilities exist between parties?",
#     "Is the distributor prohibited from using Programming and IMNTV's Marks to market, advertise, and promote the Programming in the Mobile Software Application(s?"
# ]

# # Run tests
# for q in questions:
#     print(f"\n{'='*70}")
#     print(f"Q: {q}")
#     print(f"A: {ask(q)}")

In [14]:
from src.agent.executor import run_agent
from src.agent.scope_guard import enforce_legal_scope

def ask(question: str) -> str:
    """
    Single entry point for any legal query.
    This exact function will be called from the Streamlit UI.
    """
    try:
        enforce_legal_scope(question)
    except ValueError as e:
        return str(e)
    
    return run_agent(question, session, llm)


# Test questions
questions = [
    
    "Is the distributor prohibited from using IMNTV Marks to market the software?",
    
]
# Run tests
for q in questions:
    print(f"\n{'='*70}")
    print(f"Q: {q}")
    print(f"A: {ask(q)}")


Q: Is the distributor prohibited from using IMNTV Marks to market the software?
🧠 Intent Detected: general_legal_query
📋 Plan: AgentPlan(actions=[AgentAction(type='retrieve', reason='Initial grounding required'), AgentAction(type='respond', reason=None)])
🔎 Running Retrieval Tool...
🔎 Query Used: Is the distributor prohibited from using IMNTV Marks to market the software?
   Retrieved 5 chunks.
💬 Generating grounded response with LLM...
A: No, the distributor is not prohibited from using IMNTV Marks to market the software. The Distributor may use IMNTV's Marks to market, advertise, and promote the Programming within the Mobile Software Application(s) [legal_chunk_12]. This includes featuring the Programming in product demonstrations related to the Distributor's software or subscription products [legal_chunk_12]. Additionally, the Distributor may use IMNTV logos and marks in the marketing of the Product, subject to pre-approved trademark usage policy and Brand Standards [legal_chunk_18

In [24]:
from src.agent.executor import run_agent
from src.agent.scope_guard import enforce_legal_scope

def ask(question: str) -> str:
    """
    Single entry point for any legal query.
    This exact function will be called from the Streamlit UI.
    """
    try:
        enforce_legal_scope(question)
    except ValueError as e:
        return str(e)
    
    return run_agent(question, session, llm)


# Test questions
questions = [
    
    "Under what conditions can the Distributor use IMNTV logos and trademarks in marketing the Product?",
    
]
# Run tests
for q in questions:
    print(f"\n{'='*70}")
    print(f"Q: {q}")
    print(f"A: {ask(q)}")


Q: Under what conditions can the Distributor use IMNTV logos and trademarks in marketing the Product?
🧠 Intent Detected: general_legal_query
📋 Plan: AgentPlan(actions=[AgentAction(type='respond', reason='Reusing grounded context')])
💬 Generating grounded response with LLM...
A: The Distributor may use IMNTV logos and marks in the marketing of the Product, subject to pre-approved trademark usage policy and Brand Standards [legal_chunk_18].


In [26]:
from src.agent.executor import run_agent
from src.agent.scope_guard import enforce_legal_scope

def ask(question: str) -> str:
    """
    Single entry point for any legal query.
    This exact function will be called from the Streamlit UI.
    """
    try:
        enforce_legal_scope(question)
    except ValueError as e:
        return str(e)
    
    return run_agent(question, session, llm)


# Test questions
questions = [
    
    "How many employees does IMNTV have?",
    
]
# Run tests
for q in questions:
    print(f"\n{'='*70}")
    print(f"Q: {q}")
    print(f"A: {ask(q)}")


Q: How many employees does IMNTV have?
🧠 Intent Detected: general_legal_query
📋 Plan: AgentPlan(actions=[AgentAction(type='retrieve', reason='Existing context insufficient'), AgentAction(type='respond', reason=None)])
🔎 Running Retrieval Tool...
🔎 Query Used: How many employees does IMNTV have?
   Retrieved 5 chunks.
💬 Generating grounded response with LLM...
A: The document does not contain this information.


In [25]:
from src.agent.executor import run_agent
from src.agent.scope_guard import enforce_legal_scope

def ask(question: str) -> str:
    """
    Single entry point for any legal query.
    This exact function will be called from the Streamlit UI.
    """
    try:
        enforce_legal_scope(question)
    except ValueError as e:
        return str(e)
    
    return run_agent(question, session, llm)


# Test questions
questions = [
    
    "explain caluse 2.5",
    
]
# Run tests
for q in questions:
    print(f"\n{'='*70}")
    print(f"Q: {q}")
    print(f"A: {ask(q)}")


Q: explain caluse 2.5
🧠 Intent Detected: explanation
📋 Plan: AgentPlan(actions=[AgentAction(type='retrieve', reason='Existing context insufficient'), AgentAction(type='respond', reason=None)])
🔎 Running Retrieval Tool...
🔎 Query Used: explain caluse 2.5
   Retrieved 5 chunks.
💬 Generating grounded response with LLM...
A: Clause 2.5, titled "Distributor Support and Operational Responsibilities," outlines several key duties for the Distributor [legal_chunk_14].

Under this clause, the Distributor is responsible for:
*   Providing all "middleware" programming to connect the content delivery platform to local Territory wireless networks [legal_chunk_14].
*   Performing localization of any software components to enable the program's use in the local Territory, which includes language translations or interface design changes [legal_chunk_14].
*   Acting as a liaison with local Territory wireless carriers, which involves content review to ensure it meets local broadcast standards and regulat

In [15]:
# See exactly what retrieval returns (with KG boost visible)
from src.retrieval.hybrid       import legal_hybrid_retriever
from src.kg.kg_query_engine     import kg_chunks_for_query

debug_query = "advertising policy"

# Check KG seeds first
kg_seeds = kg_chunks_for_query(session.kg, debug_query, session.chunks_df)
print(f"KG seed row indices: {kg_seeds}")

# Full retrieval
df = legal_hybrid_retriever(
    query        = debug_query,
    chunks_df    = session.chunks_df,
    embed_model  = session.embed_model,
    rerank_model = session.rerank_model,
    bm25         = session.bm25_engine,
    faiss_index  = session.faiss_engine,
    top_k        = 5,
    kg_seed_ids  = kg_seeds,
)
for _, row in df.iterrows():
    kg_flag = " ⭐ KG" if row["row_id"] in kg_seeds else ""
    print(f"\n[{row['chunk_id']}] score={row['rerank_score']:.4f}{kg_flag}")
    print(row["text"][:250])

KG seed row indices: [52, 19]
KG boosted 2 chunks.

[legal_chunk_52] score=-3.7763 ⭐ KG
Source: GLOBAL TECHNOLOGIES GROUP, INC., 10KSB, 9/28/2005





  EXHIBIT C

Programming Technical Specifications

1. REPRESENTATION OF PROGRAMMING

1.1 Programming Presentation. The following are requirements for all Programming:

(a) Each individual

[legal_chunk_13] score=-6.2122
Source: GLOBAL TECHNOLOGIES GROUP, INC., 10KSB, 9/28/2005





  software or subscription products at trade shows and conferences; (g) creating collateral for joint promotional efforts between Distributor and third parties; (h) promoting the Programm

[legal_chunk_19] score=-6.3460 ⭐ KG
3.7 Responsibility for Programming. Except as expressly set forth herein, IMNTV is solely responsible for all costs, activities, obligations and liabilities associated with: (a) obtaining all rights and licenses necessary for the authorized use and d

[legal_chunk_38] score=-7.6184
. Distributor will defend, indemnify, and hold IMNTV harml

In [16]:
# Inspect session chat history
print(f"Turns: {len(session.chat_history)}\n")
for t in session.chat_history:
    print(f"{t['role'].upper()}: {t['content'][:200]}\n")

Turns: 2

USER: Is the distributor prohibited from using IMNTV Marks to market the software?

ASSISTANT: No, the distributor is not prohibited from using IMNTV Marks to market the software. The Distributor may use IMNTV's Marks to market, advertise, and promote the Programming within the Mobile Software 



In [20]:
# Inspect document insights stored in session
fields = ["summary","parties","agreement_date","agreement_duration",
          "termination_clause","payment_terms"]
for f in fields:
    print(f"{f:25s}: {getattr(session, f)}")
print(f"{'risky_clauses':25s}: {len(session.risky_clauses or [])} found")

summary                  : An agreement for the distribution of music video content via wireless networks in China, detailing operational responsibilities, payment terms, confidentiality, and liabilities between a content provider (IMNTV) and a distributor (MobileVision).
parties                  : ['GLOBAL MUSIC INTERNATIONAL, INC. D/B/A INDEPENDENT MUSIC NETWORK (IMNTV)', 'MOBILEVISION COMMUNICATIONS LTD. (DISTRIBUTOR)']
agreement_date           : July 13, 2005
agreement_duration       : The initial term begins on the Effective Date (July 13, 2005) and ends twelve (12) months after the 'Launch' (date not specified). The agreement may be extended for additional one-year terms upon mutual agreement and satisfactory performance by both parties. There is also a pilot commercial program for two to three months where content is provided without charge.
termination_clause       : Either party may terminate the Agreement for a material breach by the other party that remains uncured after thi

In [21]:
# Document insights stored in session
for f in ["summary","parties","agreement_date","agreement_duration",
          "termination_clause","payment_terms"]:
    print(f"{f:25s}: {getattr(session, f)}")
print(f"{'risky_clauses':25s}: {len(session.risky_clauses or [])} found")
for r in (session.risky_clauses or []):
    print(f"  [{r.get('risk_level','?').upper():6}] {r.get('clause','')[:80]}")

summary                  : An agreement for the distribution of music video content via wireless networks in China, detailing operational responsibilities, payment terms, confidentiality, and liabilities between a content provider (IMNTV) and a distributor (MobileVision).
parties                  : ['GLOBAL MUSIC INTERNATIONAL, INC. D/B/A INDEPENDENT MUSIC NETWORK (IMNTV)', 'MOBILEVISION COMMUNICATIONS LTD. (DISTRIBUTOR)']
agreement_date           : July 13, 2005
agreement_duration       : The initial term begins on the Effective Date (July 13, 2005) and ends twelve (12) months after the 'Launch' (date not specified). The agreement may be extended for additional one-year terms upon mutual agreement and satisfactory performance by both parties. There is also a pilot commercial program for two to three months where content is provided without charge.
termination_clause       : Either party may terminate the Agreement for a material breach by the other party that remains uncured after thi